
### **Santa: Subs Agg + Score Track + Visual Diagnostics**

This notebook is a simple, plug-and-play “results hub” for the **Santa 2025 – Christmas Tree Packing Challenge**.
If you are iterating on solvers, tuning heuristics, or testing packing strategies across many runs, you quickly end up with a pile of submission files and no clean way to compare them. This notebook solves that problem.

### What you get

* **One place to collect everything**: load multiple submission files, merge results, and keep a clean history of your experiments
* **Side-by-side comparison**: quickly benchmark different runs and see where each approach wins/loses
* **Progress tracking**: visualize performance across *n-tree* configurations and spot regressions immediately
* **Visual diagnostics**: lightweight plots that make it obvious *why* one result beats another (not just the score)
* **Minimal assumptions**: designed to work with typical solver outputs and common community formats

### Who this is for

* You are running many experiments and want a **single dashboard** for evaluation
* You want to **compare against strong public notebooks** while keeping your own workflow organized
* You care about faster iteration: fewer spreadsheets, fewer manual checks, more insight per run

### How to use

1. Drop in your submission files (and optionally public baselines).
2. Run the notebook to aggregate, rank, and compare.
3. Use the charts to decide what to improve next.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from decimal import Decimal, getcontext
from shapely import affinity
from shapely.geometry import Polygon

# =========================
# 1) Tree geometry (same as notebook)
# =========================
getcontext().prec = 25
scale_factor = Decimal("1e15")

class ChristmasTree:
    """A single rotatable tree with fixed shape; position is (center_x, center_y), rotation is angle degrees."""
    def __init__(self, center_x="0", center_y="0", angle="0"):
        self.center_x = Decimal(str(center_x))
        self.center_y = Decimal(str(center_y))
        self.angle = Decimal(str(angle))

        trunk_w = Decimal("0.15")
        trunk_h = Decimal("0.2")
        base_w  = Decimal("0.7")
        mid_w   = Decimal("0.4")
        top_w   = Decimal("0.25")
        tip_y   = Decimal("0.8")
        tier_1_y = Decimal("0.5")
        tier_2_y = Decimal("0.25")
        base_y = Decimal("0.0")
        trunk_bottom_y = -trunk_h

        # Same vertex list as notebook (scaled up by 1e15)
        initial_polygon = Polygon(
            [
                (Decimal("0.0") * scale_factor, tip_y * scale_factor),
                (top_w / Decimal("2") * scale_factor, tier_1_y * scale_factor),
                (top_w / Decimal("4") * scale_factor, tier_1_y * scale_factor),
                (mid_w / Decimal("2") * scale_factor, tier_2_y * scale_factor),
                (mid_w / Decimal("4") * scale_factor, tier_2_y * scale_factor),
                (base_w / Decimal("2") * scale_factor, base_y * scale_factor),
                (trunk_w / Decimal("2") * scale_factor, base_y * scale_factor),
                (trunk_w / Decimal("2") * scale_factor, trunk_bottom_y * scale_factor),
                (-(trunk_w / Decimal("2")) * scale_factor, trunk_bottom_y * scale_factor),
                (-(trunk_w / Decimal("2")) * scale_factor, base_y * scale_factor),
                (-(base_w / Decimal("2")) * scale_factor, base_y * scale_factor),
                (-(mid_w / Decimal("4")) * scale_factor, tier_2_y * scale_factor),
                (-(mid_w / Decimal("2")) * scale_factor, tier_2_y * scale_factor),
                (-(top_w / Decimal("4")) * scale_factor, tier_1_y * scale_factor),
                (-(top_w / Decimal("2")) * scale_factor, tier_1_y * scale_factor),
            ]
        )

        rotated = affinity.rotate(initial_polygon, float(self.angle), origin=(0, 0))
        self.polygon = affinity.translate(
            rotated,
            xoff=float(self.center_x * scale_factor),
            yoff=float(self.center_y * scale_factor),
        )

    def clone(self):
        return ChristmasTree(self.center_x, self.center_y, self.angle)


# =========================
# 2) Helpers: parse df
# =========================
def normalize_submission_df(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure id/x/y/deg are strings; strip leading 's'; add group_n,item_id; sort."""
    d = df.copy()
    for c in ["id", "x", "y", "deg"]:
        d[c] = d[c].astype(str).str.strip()

    # allow both "s-1.23" and "-1.23"
    for c in ["x", "y", "deg"]:
        d[c] = d[c].str.lstrip("s").str.lstrip("S")

    d[["group_id", "item_id"]] = d["id"].str.split("_", n=1, expand=True)
    d["group_n"] = d["group_id"].astype(int)
    d["item_id"] = d["item_id"].astype(int)
    d = d.sort_values(["group_n", "item_id"]).reset_index(drop=True)
    return d


def build_trees_from_group(group_df: pd.DataFrame) -> list[ChristmasTree]:
    return [ChristmasTree(row["x"], row["y"], row["deg"]) for _, row in group_df.iterrows()]


# =========================
# 3) Scoring (same objective)
#    group_score = side^2 / n, total = sum over n=1..200
# =========================
def get_group_bounds_and_side(trees: list[ChristmasTree]) -> tuple[float, tuple[float, float, float, float]]:
    """
    Return:
      side (in original coordinate units),
      bounds (minx, miny, maxx, maxy) in original coordinate units
    """
    minx = miny = float("inf")
    maxx = maxy = float("-inf")
    for t in trees:
        bx0, by0, bx1, by1 = t.polygon.bounds  # scaled coordinates (1e15)
        minx = min(minx, bx0); miny = min(miny, by0)
        maxx = max(maxx, bx1); maxy = max(maxy, by1)

    # convert back to original units
    s = float(scale_factor)
    minx_, miny_, maxx_, maxy_ = minx / s, miny / s, maxx / s, maxy / s
    side = max(maxx_ - minx_, maxy_ - miny_)
    return side, (minx_, miny_, maxx_, maxy_)


def score_submission(df: pd.DataFrame, strict_count_check: bool = True) -> tuple[float, pd.DataFrame]:
    """
    Returns:
      total_score (float),
      stats_df: columns [group_n, n_rows, side, group_score]
    """
    d = normalize_submission_df(df)

    rows = []
    total = 0.0
    for n, g in d.groupby("group_n", sort=True):
        if strict_count_check and len(g) != n:
            raise ValueError(f"group {n:03d} should have {n} rows, but got {len(g)}")

        trees = build_trees_from_group(g)
        side, _bounds = get_group_bounds_and_side(trees)
        group_score = (side * side) / float(n)
        total += group_score

        rows.append(
            {
                "group_n": int(n),
                "n_rows": int(len(g)),
                "side": float(side),
                "group_score": float(group_score),
            }
        )

    stats_df = pd.DataFrame(rows).sort_values("group_n").reset_index(drop=True)
    return float(total), stats_df


# =========================
# 4) Collision check (optional sanity check)
# =========================
def has_collision(trees: list[ChristmasTree]) -> bool:
    """Collision = intersects but not touches (same as notebook). O(n^2)."""
    polys = [t.polygon for t in trees]
    m = len(polys)
    for i in range(m):
        pi = polys[i]
        for j in range(i + 1, m):
            pj = polys[j]
            if pi.intersects(pj) and (not pi.touches(pj)):
                return True
    return False


def collision_report(df: pd.DataFrame, n_list=None) -> pd.DataFrame:
    """Return per-group collision flag; default checks all groups in df."""
    d = normalize_submission_df(df)
    if n_list is None:
        n_list = sorted(d["group_n"].unique().tolist())

    out = []
    for n in n_list:
        g = d[d["group_n"] == n]
        trees = build_trees_from_group(g)
        out.append({"group_n": int(n), "has_collision": bool(has_collision(trees))})
    return pd.DataFrame(out).sort_values("group_n").reset_index(drop=True)


# =========================
# 5) Visualization
# =========================
def plot_group(df: pd.DataFrame, n: int, show_bbox: bool = True, linewidth: float = 0.8):
    d = normalize_submission_df(df)
    g = d[d["group_n"] == int(n)]
    if len(g) == 0:
        raise ValueError(f"group {n} not found in df")

    trees = build_trees_from_group(g)
    side, (minx, miny, maxx, maxy) = get_group_bounds_and_side(trees)
    group_score = (side * side) / float(n)

    plt.figure(figsize=(7, 7))
    for t in trees:
        xs, ys = t.polygon.exterior.xy
        xs = np.asarray(xs) / float(scale_factor)
        ys = np.asarray(ys) / float(scale_factor)
        plt.plot(xs, ys, linewidth=linewidth)

    if show_bbox:
        # bounding square used by score: anchored at (minx, miny), side = max(width, height)
        sqx = [minx, minx + side, minx + side, minx, minx]
        sqy = [miny, miny, miny + side, miny + side, miny]
        plt.plot(sqx, sqy, linewidth=1.2)

    plt.axis("equal")
    plt.title(f"group n={n} | side={side:.6f} | group_score={group_score:.6f}")
    plt.show()


def plot_score_curves(stats_df: pd.DataFrame):
    """Quick overview: side vs n, group_score vs n, cumulative total."""
    s = stats_df.sort_values("group_n").reset_index(drop=True)

    plt.figure(figsize=(10, 4))
    plt.plot(s["group_n"], s["side"])
    plt.xlabel("n")
    plt.ylabel("side length")
    plt.title("Side length by group")
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(s["group_n"], s["group_score"])
    plt.xlabel("n")
    plt.ylabel("group_score = side^2 / n")
    plt.title("Group score by group")
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(s["group_n"], s["group_score"].cumsum())
    plt.xlabel("n")
    plt.ylabel("cumulative score")
    plt.title("Cumulative total score (sum of group_score)")
    plt.show()



In [ ]:
# =========================================================
# 1) Load multiple submissions, score each, keep per-group stats
# =========================================================
files = [
    "/kaggle/input/santa-claude/submission_v21_1.csv",
    "/kaggle/input/santa-final-sub/bbox_sub/submi-n1000_r40_i1.csv", 
    # "/kaggle/input/why-not/submission.csv", 
    # "/kaggle/input/dumb-approach/best_submission.csv", 
    "/kaggle/input/simulated-annealing-manual-fun-tree-mover/submission.csv", 
    
]


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def normalize_submission_df(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    for c in ["id", "x", "y", "deg"]:
        d[c] = d[c].astype(str).str.strip()

    # strip leading s/S for internal use; keep as string (avoid float rounding)
    for c in ["x", "y", "deg"]:
        d[c] = d[c].str.lstrip("sS")

    d[["group_id", "item_id"]] = d["id"].str.split("_", n=1, expand=True)
    d["group_n"] = d["group_id"].astype(int)
    d["item_id"] = d["item_id"].astype(int)
    d = d.sort_values(["group_n", "item_id"]).reset_index(drop=True)
    return d



subs = {}        # key -> dict(df_raw, df_norm, total, stats)
keys = []

for f in files:
    key = Path(f).stem  # a short name for columns
    keys.append(key)

    df_raw = pd.read_csv(f)
    total_score, stats = score_submission(df_raw)  # <- 你已有的函数

    subs[key] = {
        "file": f,
        "df_raw": df_raw,
        "df_norm": normalize_submission_df(df_raw),
        "total": total_score,
        "stats": stats.sort_values("group_n").reset_index(drop=True),
    }

    print(f"TOTAL SCORE = {total_score:.8f}    {f}")

# =========================================================
# 2) Build comparison table (n=1..200): each file's group_score, best file, improvement
# =========================================================
cmp = pd.DataFrame({"group_n": np.arange(1, 201)})

for key in keys:
    st = subs[key]["stats"][["group_n", "side", "group_score"]].copy()
    st = st.rename(columns={
        "side": f"{key}_side",
        "group_score": f"{key}_score",
    })
    cmp = cmp.merge(st, on="group_n", how="left")

score_cols = [f"{k}_score" for k in keys]
cmp["best_score"] = cmp[score_cols].min(axis=1)
cmp["best_col"] = cmp[score_cols].idxmin(axis=1)
cmp["best_key"] = cmp["best_col"].str.replace("_score", "", regex=False)

# 以第一个 submission 作为 baseline，计算每组提升（你也可以换成别的）
baseline_key = keys[0]
cmp["improve_vs_baseline"] = cmp[f"{baseline_key}_score"] - cmp["best_score"]

print("\n每个 submission 获胜组数：")
print(cmp["best_key"].value_counts())

# print("\nTop 20 改进最大的组（相对 baseline）：")
# display(
#     cmp.sort_values("improve_vs_baseline", ascending=False)
#        .head(20)[["group_n", "best_key", "best_score", f"{baseline_key}_score", "improve_vs_baseline"]]
# )

# =========================================================
# 3) Construct best-of submission by picking best group from best_key
# =========================================================
rows = []
for n in range(1, 201):
    best_key = cmp.loc[cmp["group_n"] == n, "best_key"].values[0]
    dnorm = subs[best_key]["df_norm"]

    g = dnorm[dnorm["group_n"] == n].sort_values("item_id")
    if len(g) != n:
        raise ValueError(f"[{best_key}] group {n:03d} should have {n} rows, got {len(g)}")

    # rebuild id to be safe/consistent: NNN_i
    for i in range(n):
        r = g.iloc[i]
        rows.append({
            "id": f"{n:03d}_{i}",
            "x":  "s" + r["x"],
            "y":  "s" + r["y"],
            "deg":"s" + r["deg"],
        })

best_df = pd.DataFrame(rows)
best_path = "submission_bestof.csv"
best_df.to_csv(best_path, index=False)
print(f"\nWrote best-of submission to: {best_path}")

# Verify score of merged submission
best_total, best_stats = score_submission(best_df)
print(f"BEST-OF TOTAL SCORE = {best_total:.8f}")

print("\n对比总分：")
for key in keys:
    print(f"  {key:>24s}: {subs[key]['total']:.8f}")
print(f"  {'BEST-OF':>24s}: {best_total:.8f}")

# =========================================================
# 4) Visualization: per-group score gap to best-of (the closer to 0 the better)
# =========================================================
plt.figure(figsize=(12, 4))
for key in keys:
    plt.plot(cmp["group_n"], cmp[f"{key}_score"] - cmp["best_score"], label=key)
plt.axhline(0.0)
plt.xlabel("group n")
plt.ylabel("group_score - best_score (>=0)")
plt.title("Per-group gap to best-of")
plt.legend()
plt.show()

# (Optional) plot which submission wins each group (as an integer code)
mapping = {k:i for i,k in enumerate(keys)}
plt.figure(figsize=(12, 2))
plt.plot(cmp["group_n"], cmp["best_key"].map(mapping).values, ".")
plt.yticks(list(mapping.values()), list(mapping.keys()))
plt.xlabel("group n")
plt.title("Best submission per group")
plt.show()


In [ ]:
cmp.head()

In [ ]:
plot_group(best_df, 30)        


In [ ]:
plot_group(best_df, 100)        


In [ ]:
plot_group(best_df, 200)        
